LM Studio 기준

In [ ]:
#LM Studio 세팅

##INFO
#모델: LGAI-EXAONE/EXAONE-3.5-32B-Instruct-GGUF
#파일: EXAONE-3.5-32B-Instruct-Q4_K_M.gguf
#This model's API identifier: exaone-3.5-32b-instruct

##Context
#프리셋: 의미론적 분할

#Inference
#온도 0.0
#Top-k: 40
#반복 패널티: 1
#Min p 샘플링: 0.05
#Top-p: 0.95
#구조화된 출력(Structed Output-valid Json):
{
  "type": "object",
  "properties": {
    "negative_chunks": {
      "type": "array",
      "items": {
        "type": "object",
        "properties": {
          "chunk": {
            "type": "string",
            "description": "의미론적으로 연결된 AI 리스크(risk) 언급 부분 (여러 문장 가능)"
          },
          "reason": {
            "type": "string",
            "description": "판단한 이유"
          },
          "keywords": {
            "type": "array",
            "items": {
              "type": "string"
            },
            "description": "관련 키워드들"
          }
        },
        "required": [
          "chunk",
          "reason",
          "keywords"
        ]
      }
    },
    "has_negative_mention": {
      "type": "boolean",
      "description": "부정적 언급 포함 여부"
    }
  },
  "required": [
    "negative_chunks",
    "has_negative_mention"
  ]
}

##Load
#Context Length: 8096
#GPU Offload: 34/64
#CPU Thread Pool size: 4
#Evaluation Batch size: 512

실행코드

In [ ]:
import pandas as pd
import json
from openai import OpenAI
from tqdm import tqdm
import logging
from typing import List, Dict, Optional, Tuple
import re
import os
from datetime import datetime
import time

# 로깅 설정
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

class NegativeAISemanticChunkerGGUF:
    def __init__(self, api_base="http://121.132.230.74:1234/v1", model_name="exaone-3.5-32b-instruct"):
        """
        LM Studio를 통한 GGUF 모델을 사용하는 Semantic Chunker 초기화
        """
        logger.info(f"LM Studio 연결 중: {api_base}")
        logger.info(f"모델: {model_name}")

        # OpenAI 클라이언트 초기화 (LM Studio는 OpenAI API와 호환)
        self.client = OpenAI(
            base_url=api_base,
            api_key="not-needed"  # LM Studio는 API 키가 필요없음
        )
        self.model_name = model_name

        # 연결 테스트
        self.test_connection()

    def test_connection(self):
        """LM Studio 연결 테스트"""
        try:
            response = self.client.chat.completions.create(
                model=self.model_name,
                messages=[{"role": "user", "content": "Hello"}],
                max_tokens=10,
                temperature=0
            )
            logger.info("LM Studio 연결 성공")
        except Exception as e:
            logger.error(f"LM Studio 연결 실패: {e}")
            raise

    def create_prompt(self, text: str) -> str:
        """
        탭 문자를 사용하지 않도록 명시적으로 지시하는 프롬프트
        """
        prompt = f"""다음 뉴스 텍스트에서 인공지능(AI, LLM, AGI, ASI 등)의 리스크(risk)나 위험성을 언급한 부분을 의미론적 단위로 추출해주세요.

뉴스 텍스트:
{text}

다음 JSON 형식으로 응답해주세요:
{{
    "negative_chunks": [
        {{
            "chunk": "의미론적으로 연결된 AI 리스크(risk) 언급 부분 (여러 문장 가능)",
            "reason": "어떤 AI 리스크(risk)에 해당하는지 설명",
            "keywords": ["관련", "키워드들"]
        }}
    ],
    "has_negative_mention": true/false
}}

주의사항:
1. 인공지능의 리스크(risk)와 관련된 언급만 추출하세요 (위험성, 문제점, 우려, 비판 등)
2. 문맥상 중립적이거나 긍정적인 내용은 제외하세요
3. 추출된 chunk는 원문 그대로 인용하세요
4. JSON 응답 시 탭(\\t), 캐리지 리턴(\\r), 백스페이스(\\b) 등의 제어문자를 절대 사용하지 마세요
5. 들여쓰기는 공백 4개를 사용하세요
6. JSON 문법을 정확히 지켜주세요 (쉼표, 따옴표 위치 등)"""

        return prompt

    def clean_json_response(self, response: str) -> str:
        """
        JSON 응답에서 문제가 되는 문자들을 정리
        """
        # 코드 블록 제거
        response = re.sub(r'```json\s*', '', response)
        response = re.sub(r'```\s*', '', response)

        # JSON 구조는 유지하면서 문제가 되는 문자만 처리
        # 중요: JSON 내부의 실제 줄바꿈(\n)은 유지해야 함

        # 탭을 공백으로 변환
        response = response.replace('\t', '    ')

        # 캐리지 리턴 등 다른 제어 문자 제거
        response = response.replace('\r', '')
        response = response.replace('\b', '')
        response = response.replace('\f', '')
        response = response.replace('\v', '')

        return response.strip()

    def fix_json_syntax(self, json_str: str) -> str:
        """
        일반적인 JSON 문법 오류 수정
        """
        # 잘못된 쉼표 위치 수정 (배열/객체 끝의 쉼표 제거)
        json_str = re.sub(r',(\s*[}\]])', r'\1', json_str)

        # 누락된 쉼표 추가 (} 다음에 {가 오는 경우)
        json_str = re.sub(r'}\s*{', '},{', json_str)

        # 누락된 따옴표 수정 (기본적인 경우만)
        # "key": value" -> "key": "value"
        json_str = re.sub(r':\s*([^",\[\{}\]]+)"', r': "\1"', json_str)

        return json_str

    def extract_json_simple(self, response: str) -> Optional[Dict]:
        """
        개선된 JSON 추출 로직
        """
        try:
            # 1단계: 응답 정리
            cleaned_response = self.clean_json_response(response)

            # 2단계: JSON 부분만 추출
            start = cleaned_response.find('{')
            end = cleaned_response.rfind('}')

            if start == -1 or end == -1:
                logger.warning("JSON 구조를 찾을 수 없습니다")
                return None

            json_str = cleaned_response[start:end+1]

            # 3단계: 직접 파싱 시도 (줄바꿈 유지)
            try:
                result = json.loads(json_str)
                logger.info("JSON 파싱 성공")
            except json.JSONDecodeError as e:
                logger.warning(f"첫 번째 파싱 시도 실패: {e}")
                logger.debug(f"파싱 시도한 JSON: {json_str[:200]}...")

                # 4단계: JSON 문법 오류 수정 후 재시도
                json_str = self.fix_json_syntax(json_str)

                try:
                    result = json.loads(json_str)
                    logger.info("수정 후 JSON 파싱 성공")
                except json.JSONDecodeError as e2:
                    logger.warning(f"두 번째 파싱 시도 실패: {e2}")
                    # 더 공격적인 정리 시도
                    json_str = self.aggressive_clean(json_str)
                    result = json.loads(json_str)

            # 5단계: 결과 검증 및 기본값 설정
            if "has_negative_mention" not in result:
                result["has_negative_mention"] = len(result.get("negative_chunks", [])) > 0
            if "negative_chunks" not in result:
                result["negative_chunks"] = []

            return result

        except Exception as e:
            logger.error(f"JSON 파싱 실패: {e}")
            logger.debug(f"문제가 된 응답: {response[:500]}...")
            return self.manual_parse_fallback(response)

    def aggressive_clean(self, json_str: str) -> str:
        """
        더 공격적인 JSON 정리 (최후의 수단)
        """
        # 모든 공백 문자를 단일 공백으로
        json_str = ' '.join(json_str.split())

        # 잘못된 이스케이프 문자 제거
        json_str = json_str.replace('\\t', ' ')
        json_str = json_str.replace('\\r', ' ')
        json_str = json_str.replace('\\n', ' ')
        json_str = json_str.replace('\\b', ' ')

        # 중복된 쉼표 제거
        json_str = re.sub(r',+', ',', json_str)

        # 문자열 내의 따옴표 이스케이프
        # "text "quoted" text" -> "text \"quoted\" text"
        def escape_quotes_in_strings(match):
            content = match.group(1)
            # 내부 따옴표 이스케이프
            content = content.replace('"', '\\"')
            return f'"{content}"'

        # 매우 신중하게 처리
        parts = []
        in_string = False
        i = 0
        while i < len(json_str):
            if json_str[i] == '"' and (i == 0 or json_str[i-1] != '\\'):
                in_string = not in_string
            parts.append(json_str[i])
            i += 1

        return ''.join(parts)

    def manual_parse_fallback(self, response: str) -> Dict:
        """
        개선된 수동 파싱 (정규식 기반)
        """
        try:
            chunks = []

            # chunk 패턴 찾기 (더 유연한 정규식)
            chunk_pattern = r'"chunk"\s*:\s*"([^"]*(?:\\"[^"]*)*)"'
            reason_pattern = r'"reason"\s*:\s*"([^"]*(?:\\"[^"]*)*)"'
            keywords_pattern = r'"keywords"\s*:\s*\[([^\]]*)\]'

            chunk_matches = re.findall(chunk_pattern, response, re.DOTALL)
            reason_matches = re.findall(reason_pattern, response, re.DOTALL)
            keywords_matches = re.findall(keywords_pattern, response, re.DOTALL)

            for i, chunk in enumerate(chunk_matches):
                # 이스케이프 문자 복원
                chunk = chunk.replace('\\"', '"')
                chunk = chunk.strip()

                reason = ""
                if i < len(reason_matches):
                    reason = reason_matches[i].replace('\\"', '"').strip()

                keywords = []
                if i < len(keywords_matches):
                    # 키워드 파싱
                    kw_str = keywords_matches[i]
                    kw_matches = re.findall(r'"([^"]*)"', kw_str)
                    keywords = [kw.strip() for kw in kw_matches]

                if chunk:  # 빈 chunk는 제외
                    chunks.append({
                        "chunk": chunk,
                        "reason": reason,
                        "keywords": keywords
                    })

            # has_negative_mention 찾기
            has_negative = bool(chunks)
            neg_mention_match = re.search(r'"has_negative_mention"\s*:\s*(true|false)', response, re.IGNORECASE)
            if neg_mention_match:
                has_negative = neg_mention_match.group(1).lower() == 'true'

            return {
                "negative_chunks": chunks,
                "has_negative_mention": has_negative
            }

        except Exception as e:
            logger.error(f"수동 파싱도 실패: {e}")
            return {
                "negative_chunks": [],
                "has_negative_mention": False,
                "parse_error": True
            }

    def analyze_single_text(self, text: str, max_retries: int = 3) -> Dict:
        """
        단일 텍스트 분석 (개선된 재시도 로직)
        """
        prompt = self.create_prompt(text)

        for attempt in range(max_retries):
            try:
                # LM Studio API 호출
                response = self.client.chat.completions.create(
                    model=self.model_name,
                    messages=[
                        {
                            "role": "system",
                            "content": "당신은 인공지능의 위험성을 연구하는 컴퓨터공학자입니다. JSON 응답 시 탭 문자(\\t)나 다른 제어 문자를 사용하지 마세요. 들여쓰기는 공백으로만 하세요."
                        },
                        {"role": "user", "content": prompt}
                    ],
                    temperature=0.0,
                    max_tokens=8096,
                    stream=False
                )

                # 응답 추출
                response_text = response.choices[0].message.content.strip()

                # 디버깅을 위한 로그 (필요시 활성화)
                # logger.debug(f"원본 응답: {response_text[:200]}...")

                # JSON 추출
                result = self.extract_json_simple(response_text)

                if result is not None and not result.get("parse_error", False):
                    return result
                else:
                    logger.warning(f"시도 {attempt + 1}/{max_retries}: JSON 파싱 실패")
                    if attempt < max_retries - 1:
                        time.sleep(1)

            except Exception as e:
                logger.error(f"시도 {attempt + 1}/{max_retries}: API 호출 오류 - {e}")
                if attempt < max_retries - 1:
                    time.sleep(2)

        # 모든 시도 실패 시 기본값 반환
        return {
            "negative_chunks": [],
            "has_negative_mention": False,
            "parse_error": True
        }

    def analyze_batch(self, texts: List[str]) -> List[Dict]:
        """
        배치 텍스트 분석
        """
        results = []

        for text in tqdm(texts, desc="Analyzing batch"):
            result = self.analyze_single_text(text)
            results.append(result)
            time.sleep(0.1)  # API 부하 방지

        return results

    def save_checkpoint_csv(self, checkpoint_path: str, all_results: List[Dict], df_columns: List[str]):
        """
        체크포인트를 CSV 형식으로 저장
        """
        if all_results:
            # 결과를 DataFrame으로 변환
            result_df = pd.DataFrame(all_results)

            # 열 순서 정리 (원본 데이터 열 + 분석 결과 열)
            analysis_columns = ["chunk", "reason", "keywords", "processed_at"]
            ordered_columns = [col for col in df_columns if col in result_df.columns] + analysis_columns

            # 존재하는 열만 선택
            existing_columns = [col for col in ordered_columns if col in result_df.columns]
            result_df = result_df[existing_columns]

            # CSV로 저장
            result_df.to_csv(checkpoint_path, index=False, encoding='utf-8-sig')
            logger.info(f"체크포인트 CSV 저장: {checkpoint_path} (총 {len(result_df)}개 행)")
        else:
            # 빈 결과인 경우 헤더만 저장
            columns = df_columns + ["chunk", "reason", "keywords", "processed_at"]
            empty_df = pd.DataFrame(columns=columns)
            empty_df.to_csv(checkpoint_path, index=False, encoding='utf-8-sig')
            logger.info(f"빈 체크포인트 CSV 저장: {checkpoint_path}")

    def load_checkpoint_csv(self, checkpoint_path: str) -> Tuple[List[int], List[Dict]]:
        """
        CSV 체크포인트 파일 로드
        """
        if os.path.exists(checkpoint_path):
            try:
                df = pd.read_csv(checkpoint_path, encoding='utf-8-sig')
                if len(df) > 0:
                    # 이미 처리된 인덱스 추출 (원본 데이터의 인덱스가 있다면)
                    processed_indices = []
                    if 'original_index' in df.columns:
                        processed_indices = df['original_index'].tolist()

                    # 결과 데이터를 딕셔너리 리스트로 변환
                    all_results = df.to_dict('records')

                    logger.info(f"체크포인트 CSV 로드: {checkpoint_path} (총 {len(all_results)}개 행)")
                    return processed_indices, all_results
                else:
                    logger.info(f"빈 체크포인트 파일 로드: {checkpoint_path}")
                    return [], []
            except Exception as e:
                logger.error(f"체크포인트 로드 실패: {e}")
                return [], []
        return [], []

    def create_processed_index_file(self, checkpoint_path: str) -> str:
        """
        처리된 인덱스를 별도로 저장하는 파일 경로 생성
        """
        base_name = os.path.splitext(checkpoint_path)[0]
        return f"{base_name}_processed_indices.txt"

    def save_processed_indices(self, index_file_path: str, processed_indices: List[int]):
        """
        처리된 인덱스를 텍스트 파일로 저장
        """
        with open(index_file_path, 'w') as f:
            for idx in processed_indices:
                f.write(f"{idx}\n")

    def load_processed_indices(self, index_file_path: str) -> List[int]:
        """
        처리된 인덱스를 텍스트 파일에서 로드
        """
        if os.path.exists(index_file_path):
            with open(index_file_path, 'r') as f:
                return [int(line.strip()) for line in f if line.strip()]
        return []

    def process_csv_with_checkpoint(self, input_file: str, output_file: str,
                                  text_column: str = 'text', batch_size: int = 2,
                                  checkpoint_interval: int = 10):
        """
        체크포인트를 활용한 CSV 파일 처리 (CSV 형식 체크포인트)
        """
        logger.info(f"CSV 파일 읽는 중: {input_file}")
        df = pd.read_csv(input_file)

        if text_column not in df.columns:
            raise ValueError(f"'{text_column}' 열이 CSV 파일에 없습니다")

        # 체크포인트 경로 설정
        base_name = os.path.splitext(output_file)[0]
        checkpoint_path = f"{base_name}_checkpoint.csv"
        index_file_path = self.create_processed_index_file(checkpoint_path)

        # 체크포인트 로드
        processed_indices = self.load_processed_indices(index_file_path)
        _, all_results = self.load_checkpoint_csv(checkpoint_path)

        processed_set = set(processed_indices)

        logger.info(f"총 {len(df)}개의 행 처리 시작 (이미 처리된 행: {len(processed_set)}개)")
        logger.info(f"체크포인트 파일: {checkpoint_path}")

        # 배치 처리를 위한 버퍼
        batch_indices = []
        batch_texts = []

        # 진행률 표시
        pbar = tqdm(df.iterrows(), total=len(df), desc="Processing rows")

        for idx, row in pbar:
            # 이미 처리된 행은 건너뛰기
            if idx in processed_set:
                continue

            text = row[text_column]

            if pd.isna(text) or text.strip() == "":
                processed_indices.append(idx)
                continue

            # 배치에 추가
            batch_indices.append(idx)
            batch_texts.append(text)

            # 배치가 가득 찼거나 마지막 행인 경우 처리
            if len(batch_texts) == batch_size or idx == len(df) - 1:
                # 배치 분석
                batch_results = self.analyze_batch(batch_texts)

                # 결과 저장
                for i, (batch_idx, result) in enumerate(zip(batch_indices, batch_results)):
                    if not result.get("parse_error", False) and result.get("has_negative_mention", False):
                        for chunk_info in result.get("negative_chunks", []):
                            # 원본 데이터의 모든 열을 포함
                            result_row = df.iloc[batch_idx].to_dict()

                            # 원본 인덱스 추가
                            result_row['original_index'] = batch_idx

                            # 청크 정보 추가
                            keywords = chunk_info.get("keywords", [])
                            if isinstance(keywords, list):
                                keywords_str = ",".join(keywords)
                            else:
                                keywords_str = str(keywords)

                            result_row.update({
                                "chunk": chunk_info.get("chunk", ""),
                                "reason": chunk_info.get("reason", ""),
                                "keywords": keywords_str,
                                "processed_at": datetime.now().isoformat()
                            })

                            all_results.append(result_row)

                    processed_indices.append(batch_idx)

                # 배치 초기화
                batch_indices = []
                batch_texts = []

                # 주기적으로 체크포인트 저장
                if len(processed_indices) % checkpoint_interval == 0:
                    self.save_checkpoint_csv(checkpoint_path, all_results, list(df.columns))
                    self.save_processed_indices(index_file_path, processed_indices)

                # 진행률 업데이트
                pbar.set_postfix({'processed': len(processed_indices), 'chunks': len(all_results)})

        # 최종 결과 저장
        logger.info(f"총 {len(all_results)}개의 부정적 chunk 발견")

        if all_results:
            result_df = pd.DataFrame(all_results)
            result_df.to_csv(output_file, index=False, encoding='utf-8-sig')
            logger.info(f"최종 결과 저장 완료: {output_file}")
        else:
            logger.warning("부정적 언급이 발견되지 않았습니다")
            # 빈 DataFrame 저장
            empty_df = pd.DataFrame(columns=list(df.columns) + ["chunk", "reason", "keywords", "processed_at"])
            empty_df.to_csv(output_file, index=False, encoding='utf-8-sig')

        # 최종 체크포인트 저장
        self.save_checkpoint_csv(checkpoint_path, all_results, list(df.columns))
        self.save_processed_indices(index_file_path, processed_indices)

        logger.info(f"체크포인트 파일이 CSV 형식으로 저장되었습니다: {checkpoint_path}")
        logger.info("처리 완료 후에도 체크포인트 파일을 확인할 수 있습니다.")

        return all_results


# ========== 실행 코드 ==========

def run_analysis(input_csv_path: str, output_csv_path: str, text_column: str = "text",
                batch_size: int = 2, api_base: str = "http://121.132.230.74:1234/v1",
                model_name: str = "exaone-3.5-32b-instruct"):
    """
    분석 실행 함수
    """

    # 경로 유효성 검사
    if not os.path.exists(input_csv_path):
        logger.error(f"입력 파일을 찾을 수 없습니다: {input_csv_path}")
        return None

    # 출력 디렉토리 생성
    output_dir = os.path.dirname(output_csv_path)
    if output_dir and not os.path.exists(output_dir):
        os.makedirs(output_dir)
        logger.info(f"출력 디렉토리 생성: {output_dir}")

    # 타임스탬프 추가
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    base_name, ext = os.path.splitext(output_csv_path)
    output_path_with_timestamp = f"{base_name}_{timestamp}{ext}"

    logger.info("=== Negative AI Semantic Chunking (GGUF) 시작 ===")
    logger.info(f"입력 파일: {input_csv_path}")
    logger.info(f"출력 파일: {output_path_with_timestamp}")
    logger.info(f"체크포인트 파일: {base_name}_{timestamp}_checkpoint.csv")
    logger.info(f"텍스트 열: {text_column}")
    logger.info(f"배치 크기: {batch_size}")
    logger.info(f"API 엔드포인트: {api_base}")
    logger.info(f"모델: {model_name}")

    try:
        # Chunker 초기화
        chunker = NegativeAISemanticChunkerGGUF(api_base=api_base, model_name=model_name)

        # CSV 파일 처리
        results = chunker.process_csv_with_checkpoint(
            input_file=input_csv_path,
            output_file=output_path_with_timestamp,
            text_column=text_column,
            batch_size=batch_size
        )

        logger.info("=== 처리 완료 ===")
        logger.info(f"최종 결과 파일: {output_path_with_timestamp}")
        logger.info(f"체크포인트 파일: {base_name}_{timestamp}_checkpoint.csv")

        return results

    except Exception as e:
        logger.error(f"처리 중 오류 발생: {str(e)}")
        raise e


# 사용 예제(5번과 6번 결합해서 데이터셋 구성할 것)
if __name__ == "__main__":
    # 파일 경로 설정
    INPUT_CSV_PATH = "/mnt/c/Users/com/Desktop/ai컴기말/0. Data/2. preprocesseddata/Ai_전처리데이터4.csv"
    OUTPUT_CSV_PATH = "/mnt/c/Users/com/Desktop/ai컴기말/0. Data/3. semanticChunkeddata/Ai_ChunkedData6.csv"

    # 분석 실행
    results = run_analysis(
        input_csv_path=INPUT_CSV_PATH,
        output_csv_path=OUTPUT_CSV_PATH,
        text_column="text",
        batch_size=2,
        api_base="http://121.132.230.74:1234/v1",
        model_name="exaone-3.5-32b-instruct"
    )

Appendix

In [ ]:
#Model Inforamtion
Model = LGAI-EXAONE/EXAONE-3.5-32B-Instruct-GGUF
File = EXAONE-3.5-32B-Instruct-Q4_K_M.gguf
Format = GGUF
Quantization = Q4_K_M
Arch = exaone
Domain = llm
Size on disk = 19.34GB

decdoding

In [ ]:
#decoding
temperature=0
top_k_sampling=40
repetition_penalty=1.0
min_p_sampling-0.05
top_p=0.95
max_tokens=8196

Inference/Structured Output/

In [ ]:
{
  "type": "object",
  "properties": {
    "negative_chunks": {
      "type": "array",
      "items": {
        "type": "object",
        "properties": {
          "chunk": {
            "type": "string",
            "description": "의미론적으로 연결된 AI 리스크(risk) 언급 부분 (여러 문장 가능)"
          },
          "reason": {
            "type": "string",
            "description": "판단한 이유"
          },
          "keywords": {
            "type": "array",
            "items": {
              "type": "string"
            },
            "description": "관련 키워드들"
          }
        },
        "required": [
          "chunk",
          "reason",
          "keywords"
        ]
      }
    },
    "has_negative_mention": {
      "type": "boolean",
      "description": "부정적 언급 포함 여부"
    }
  },
  "required": [
    "negative_chunks",
    "has_negative_mention"
  ]
}